# 1.3 Validate data quality and manage bias

**Story:** A new month of claims (`claims_v2`) arrives and it is messy: missing values, duplicates,
impossible numbers. Fraud is rare, the fairness team asks whether the data treats age groups, genders
and regions equally, and the fine-tuning file for the claims assistant has never been checked.
You validate, clean, balance and screen the data before anything is trained on it.

**Task statement 1.3:** Validate data quality and manage bias.

| Skill | Topic | Status |
|---|---|---|
| 1.3.1 | Validate data quality (Glue Data Quality, DataBrew profile) | ✅ |
| 1.3.2 | Label and annotate data (Ground Truth) | 🔒 needs a human workforce; manifests run |
| 1.3.3 | Bias in data: splitting, shuffling, Clarify pre-training bias | ✅ |
| 1.3.4 | Bias metrics across numeric, text and image data | ✅ |
| 1.3.5 | Class imbalance in numeric, text and image data | ✅ |
| 1.3.6 | AI training-data integrity (pair validation, content safety) | ✅ |
| 1.3.7 | Clean data (outliers, missing values, duplicates) | ✅ |

💰 Cost of a full run: well under USD 1 (Glue Data Quality run, DataBrew profile job, one Clarify job on `ml.m5.large`).

## Key points for the exam

- **Glue Data Quality**: rules written in **DQDL** (`IsComplete`, `IsUnique`, `ColumnValues`, `Completeness`, `RowCount` ...), run on a catalog table or inside a Glue job; gives a score and pass/fail per rule; can recommend rules.
- **DataBrew profile job**: statistics per column (missing, distinct, distribution, correlations) without code.
- **Clarify pre-training bias** looks at the **data only**. **CI** (class imbalance) = is one group under-represented?
  **DPL** (difference in proportions of labels) = do groups get the positive label at different rates? Post-training metrics (DPPL, DI ...) need a model.
- **Class imbalance fixes**: resample (oversample minority, undersample majority, SMOTE), class weights (`scale_pos_weight` in XGBoost), more data / augmentation, and the right metric (recall, precision, F1, PR-AUC, not accuracy).
- **Resample and augment the training set only.** Validation and test keep the real distribution.
- **Split before** fitting anything; **shuffle** unless the data is a time series; **stratify** on the label; keep groups (same customer) on one side.
- **Ground Truth**: human labeling with a private, vendor or Mechanical Turk workforce; input and output are **manifest** files (JSON Lines).
- **Fine-tuning data integrity**: check schema, empties, duplicates, length, PII and harmful content before training; garbage or poisoned pairs become model behaviour.

| Missing-value fix | When |
|---|---|
| Drop rows / columns | Few rows affected, or a column is mostly empty |
| Mean | Numeric, roughly symmetric |
| Median | Numeric with outliers or skew |
| Mode / constant / "unknown" | Categorical |
| Model-based (k-NN, MICE) | Values depend on other columns and accuracy matters |

## Setup

In [ ]:
import sys; sys.path.append("..")
from common import utils as U, catchup as C
print(U.REGION, U.bucket())

Catch-up: raw files in S3 and the training files from notebook 1.2a (built now if missing).

In [ ]:
RAW = C.ensure_raw_data()
state = C.ensure_training_data()
print(state["train_s3"])

Common imports and names.

In [ ]:
import io, json, os, re, time
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from common import features as F

BUCKET, ROLE = U.bucket(), U.role_arn()
TAG_MAP = {t["Key"]: t["Value"] for t in U.TAGS}
s3 = U.client("s3")

def exists(fn, **kwargs):
    """True if a describe/get call works, False if the resource is not there."""
    try:
        fn(**kwargs)
        return True
    except ClientError:
        return False

raw = U.read_parquet(f"{RAW}/claims_v2.parquet")
print(raw.shape)

## 1.3.1 Validate data quality

- Write down what "good data" means as **rules**, run them automatically on every new batch, and stop the pipeline when they fail.
- **Glue Data Quality** = rules (DQDL) on Glue tables or in ETL jobs. **DataBrew** = profile jobs and data quality rulesets, no code.
  **Data Wrangler** has a Data Quality and Insights report; **SageMaker Model Monitor** checks data quality of live endpoints (Domain 4).

### Put `claims_v2` in the lake and register a Glue table
Glue Data Quality evaluates a **Data Catalog table**. We copy the file to its own lake folder first.

In [ ]:
LAKE_KEY = "safedrive/lake/claims_v2/claims_v2.parquet"
s3.copy_object(Bucket=BUCKET, Key=LAKE_KEY, CopySource={"Bucket": BUCKET, "Key": f"{RAW}/claims_v2.parquet"})
glue = U.client("glue")
DB, TABLE = "safedrive", "claims_v2"
if not exists(glue.get_database, Name=DB):
    glue.create_database(DatabaseInput={"Name": DB, "Description": "SafeDrive data lake"}, Tags=TAG_MAP)
print(glue.get_database(Name=DB)["Database"]["Name"], "|", U.s3_uri(LAKE_KEY))

Create the table definition by hand (a crawler would do the same, as in notebook 1.1). Column types come from the pandas dtypes.

In [ ]:
type_map = {"int64": "bigint", "float64": "double"}
columns = [{"Name": c, "Type": type_map.get(str(t), "string")} for c, t in raw.dtypes.items()]
hive = "org.apache.hadoop.hive.ql.io.parquet"
if not exists(glue.get_table, DatabaseName=DB, Name=TABLE):
    glue.create_table(DatabaseName=DB, TableInput={
        "Name": TABLE, "TableType": "EXTERNAL_TABLE", "Parameters": {"classification": "parquet"},
        "StorageDescriptor": {
            "Columns": columns, "Location": U.s3_uri("safedrive/lake/claims_v2/"),
            "InputFormat": f"{hive}.MapredParquetInputFormat",
            "OutputFormat": f"{hive}.MapredParquetOutputFormat",
            "SerdeInfo": {"SerializationLibrary": f"{hive}.serde.ParquetHiveSerDe"}}})
print([f"{c['Name']}:{c['Type']}" for c in columns])

### Write the rules in DQDL
Each line is one rule. Read them as sentences: "claim_id is complete", "claim_type is one of these five values".

In [ ]:
RULESET = "safedrive-claims-v2-rules"
DQDL = """Rules = [
    RowCount > 1000,
    IsComplete "claim_id",
    IsUnique "claim_id",
    Completeness "claim_amount" > 0.99,
    Completeness "police_report" > 0.99,
    ColumnValues "claim_type" in ["collision", "theft", "glass", "fire", "third_party"],
    ColumnValues "report_delay_days" >= 0,
    ColumnValues "claim_amount" <= 200000,
    ColumnValues "incident_hour" <= 23,
    Mean "is_fraud" between 0.01 and 0.20
]"""
print(DQDL)

### Create the ruleset
The ruleset is stored in Glue and linked to the table. An old ruleset with the same name is replaced.

In [ ]:
if exists(glue.get_data_quality_ruleset, Name=RULESET):
    glue.delete_data_quality_ruleset(Name=RULESET)
glue.create_data_quality_ruleset(Name=RULESET, Ruleset=DQDL, Tags=TAG_MAP,
                                 Description="Quality rules for monthly claims",
                                 TargetTable={"TableName": TABLE, "DatabaseName": DB})
print(glue.get_data_quality_ruleset(Name=RULESET)["Name"])

### Start an evaluation run
Glue starts a small Spark job (2 workers) that checks every rule against the table.

In [ ]:
dq_run = glue.start_data_quality_ruleset_evaluation_run(
    DataSource={"GlueTable": {"DatabaseName": DB, "TableName": TABLE}},
    Role=ROLE, NumberOfWorkers=2, Timeout=20, RulesetNames=[RULESET])["RunId"]
print(dq_run)

Wait for the run (2 to 4 minutes).

In [ ]:
U.wait_until(lambda: glue.get_data_quality_ruleset_evaluation_run(RunId=dq_run)["Status"],
             ok=("SUCCEEDED",), bad=("FAILED", "STOPPED", "TIMEOUT"), delay=20, label="data quality run")

### Read the result
One overall **score** (share of rules that passed) and a PASS / FAIL with a message for each rule.
Which problems did the rules catch?

In [ ]:
result_id = glue.get_data_quality_ruleset_evaluation_run(RunId=dq_run)["ResultIds"][0]
dq = glue.get_data_quality_result(ResultId=result_id)
print("score:", dq["Score"])
pd.set_option("display.max_colwidth", 110)
pd.DataFrame(dq["RuleResults"])[["Name", "Description", "Result", "EvaluationMessage"]]

> 🖥️ **Console check:** *AWS Glue → Data Catalog → Tables → `claims_v2` → Data quality tab*.
> The ruleset, the run history and the score are shown there, with each rule marked passed or failed. This is where a data steward would look, without any code.

### DataBrew: create a dataset and a profile job
A **profile job** computes statistics for every column. No rules needed: use it to *discover* what is wrong.

In [ ]:
brew = U.client("databrew")
BREW_DS, PROFILE_JOB = "safedrive-claims-v2", "safedrive-claims-v2-profile"
PROFILE_OUT = "safedrive/dq/databrew-profile/"
if not exists(brew.describe_dataset, Name=BREW_DS):
    brew.create_dataset(Name=BREW_DS, Format="PARQUET", Tags=TAG_MAP,
                        Input={"S3InputDefinition": {"Bucket": BUCKET, "Key": f"{RAW}/claims_v2.parquet"}})
if exists(brew.describe_job, Name=PROFILE_JOB):
    brew.delete_job(Name=PROFILE_JOB)
brew.create_profile_job(Name=PROFILE_JOB, DatasetName=BREW_DS, RoleArn=ROLE, MaxCapacity=2, Timeout=30,
                        OutputLocation={"Bucket": BUCKET, "Key": PROFILE_OUT},
                        JobSample={"Mode": "FULL_DATASET"}, Tags=TAG_MAP)
print("created", PROFILE_JOB)

### DataBrew: run the profile job and wait
About 3 to 5 minutes.

In [ ]:
profile_run = brew.start_job_run(Name=PROFILE_JOB)["RunId"]
U.wait_until(lambda: brew.describe_job_run(Name=PROFILE_JOB, RunId=profile_run)["State"],
             ok=("SUCCEEDED",), bad=("FAILED", "STOPPED", "TIMEOUT"), delay=30, label="profile job")

### DataBrew: read the profile
The job writes a JSON report to S3 (the DataBrew console shows it as charts). Look at missing and distinct counts per column.

In [ ]:
profile_key = sorted(k for k in U.list_keys(PROFILE_OUT) if k.endswith(".json"))[-1]
profile = json.loads(U.read_bytes(profile_key))
print("top-level keys:", list(profile)[:8], "| duplicate rows:", profile.get("duplicateRowsCount"))
pd.DataFrame([{"column": c.get("name"), "type": c.get("type"), "missing": c.get("missingValuesCount"),
               "distinct": c.get("distinctValuesCount"), "max": c.get("max")} for c in profile.get("columns", [])])

> 🖥️ **Console check:** *AWS Glue DataBrew → Datasets → `safedrive-claims-v2` → Data profile overview*.
> The same JSON as charts: missing and duplicate counts, value distributions and a correlation matrix. The *Column statistics* tab shows outliers per column.

## 1.3.7 Clean data

The checks found the problems. Now we look at each one in pandas and fix it. The steps end in exactly
what `F.clean_claims` does (the function the feature pipeline uses).

### Missing values
Count nulls per column. Three columns are affected.

In [ ]:
nulls = raw.isna().sum()
pd.DataFrame({"missing": nulls, "percent": (nulls / len(raw) * 100).round(1)})[nulls > 0]

### Duplicates
The same `claim_id` appears more than once (a file was loaded twice). Duplicates inflate metrics and can
leak between train and test.

In [ ]:
print("rows:", len(raw), "| unique claim_id:", raw["claim_id"].nunique(),
      "| duplicated rows:", raw.duplicated(subset="claim_id").sum())
raw[raw.duplicated(subset="claim_id", keep=False)].sort_values("claim_id").head(4)

### Bad categories and impossible values
`"COLLISION "` (upper case, trailing space) is the same thing as `collision`. A report delay cannot be negative.

In [ ]:
print(raw["claim_type"].value_counts().to_dict())
print("negative report_delay_days:", (raw["report_delay_days"] < 0).sum())

### Outliers: look at the distribution
The maximum is far away from the 75% and 99% values. The mean is pulled up; the median is not.

In [ ]:
raw["claim_amount"].describe(percentiles=[0.5, 0.75, 0.99]).round(1)

### Outliers: two detection rules
- **IQR rule**: outside `Q3 + 1.5 × IQR` (flags the natural long tail too).
- **Z-score**: more than 3 standard deviations from the mean (the outliers themselves inflate the standard deviation).

In [ ]:
amount = raw["claim_amount"]
q1, q3 = amount.quantile([0.25, 0.75])
iqr_outliers = amount > q3 + 1.5 * (q3 - q1)
z_outliers = ((amount - amount.mean()) / amount.std()).abs() > 3
print("IQR rule:", iqr_outliers.sum(), "| z-score rule:", z_outliers.sum(), "| amount = 9,999,999:", (amount == 9_999_999).sum())

### Fix 1: remove duplicates
Keep the first row of each `claim_id`.

In [ ]:
clean = raw.drop_duplicates(subset="claim_id").copy()
print(len(raw), "->", len(clean), "rows")

### Fix 2: standardize categories
Trim spaces and lower-case.

In [ ]:
clean["claim_type"] = clean["claim_type"].str.strip().str.lower()
clean["claim_type"].value_counts().to_dict()

### Fix 3: repair impossible values
A negative delay is a data-entry error; set it to 0 (the business rule chosen by SafeDrive).

In [ ]:
clean.loc[clean["report_delay_days"] < 0, "report_delay_days"] = 0
print("negative delays left:", (clean["report_delay_days"] < 0).sum())

### Fix 4: cap outliers, then impute missing amounts
- **Capping (winsorizing)** at the 99th percentile keeps the row but limits the extreme value.
- Missing amounts get the **median** (robust to the outliers; the mean would be far too high).

In [ ]:
cap, median = clean["claim_amount"].quantile(0.99), clean["claim_amount"].median()
print("cap:", round(cap, 2), "| median:", round(median, 2), "| mean before:", round(clean["claim_amount"].mean(), 2))
clean["claim_amount"] = clean["claim_amount"].clip(upper=cap).fillna(median)
print("mean after:", round(clean["claim_amount"].mean(), 2), "| max after:", round(clean["claim_amount"].max(), 2))

### Fix 5: impute the other missing values
`incident_hour` gets a neutral constant (12 = midday); `police_report` gets `"N"` (no report on file).

In [ ]:
clean["incident_hour"] = clean["incident_hour"].fillna(12)
clean["police_report"] = clean["police_report"].fillna("N")
print("missing values left:", int(clean.isna().sum().sum()))

### Compare with `F.clean_claims` and save
Our step-by-step result must be identical to the shared function. Then write the clean file for later notebooks.

In [ ]:
print("same as F.clean_claims:", clean.equals(F.clean_claims(raw)))
CLEAN_URI = U.write_df(clean, "safedrive/processed/claims_v2_clean.parquet", fmt="parquet")
U.save_state(claims_v2_clean_s3=CLEAN_URI)
print(CLEAN_URI, clean.shape)

## 1.3.3 Identify and mitigate bias in data

| Source of bias | Example at SafeDrive | Mitigation |
|---|---|---|
| Sampling / selection | Few claims from the West region | Collect more, resample, reweight |
| Ordering | File sorted by claim type or date, split without shuffling | Shuffle before splitting (not for time series) |
| Label imbalance | 5% fraud | Stratified split, class weights, resampling (1.3.5) |
| Leakage between splits | Same customer in train and test | Group-aware split, deduplicate first |
| Label disparity between groups | One group is labelled fraud more often | Measure (Clarify DPL), review labels, reweight |

### Why shuffle
Suppose the export is sorted by `claim_type`. Taking the last 20% as the test set **without shuffling**
gives a test set with only two claim types.

In [ ]:
sorted_df = clean.sort_values("claim_type").reset_index(drop=True)
n_train = int(len(sorted_df) * 0.8)
shuffled = sorted_df.sample(frac=1, random_state=42)
pd.DataFrame({"test set, no shuffle": sorted_df.iloc[n_train:]["claim_type"].value_counts(),
              "test set, shuffled": shuffled.iloc[n_train:]["claim_type"].value_counts()}).fillna(0).astype(int)

### Why stratify
With a rare label, a random split gives a different fraud rate each time. A **stratified** split keeps it constant.
Five small test sets, random vs stratified:

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

claims = U.read_csv(f"{RAW}/claims.csv")
policies = U.read_csv(f"{RAW}/policies.csv")
rates = {"random": [], "stratified": []}
for seed in range(5):
    rates["random"].append(train_test_split(claims, test_size=0.02, random_state=seed)[1]["is_fraud"].mean())
    rates["stratified"].append(train_test_split(claims, test_size=0.02, random_state=seed,
                                                stratify=claims["is_fraud"])[1]["is_fraud"].mean())
pd.DataFrame(rates).round(4).T

### Group leakage
One policy can have several claims. In a random split most test policies also appear in training, so the
model can "remember" customers. `GroupShuffleSplit` keeps each policy on one side.

In [ ]:
tr, te = train_test_split(claims, test_size=0.2, random_state=0)
print("random split : policies in both sets =", len(set(tr["policy_id"]) & set(te["policy_id"])))
tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(claims, groups=claims["policy_id"]))
print("grouped split: policies in both sets =",
      len(set(claims.iloc[tr_idx]["policy_id"]) & set(claims.iloc[te_idx]["policy_id"])))

**📘 Concept: Facets and pre-training bias metrics (CI, DPL)**

Clarify vocabulary first:
- **Facet** = the sensitive attribute to check (age group, gender, region). The value you worry about is the *disadvantaged* group **d**; everyone else is group **a**.
- **Positive (favourable) label** = the outcome people want (loan approved; here: claim *not* flagged as fraud).

**Pre-training** metrics need only the dataset, so they can run before any model exists:

| Metric | Question it answers | Reading |
|---|---|---|
| **CI** class imbalance | Does group d have far fewer rows than group a? | −1 to +1. Near 0 = balanced; positive = d is under-represented |
| **DPL** difference in proportions of labels | Does group d get the favourable label less often? | Near 0 = equal; positive = group a gets it more often |
| KL, JS, and others | How different are the label distributions of the two groups? | 0 = identical |

Why both matter: with high **CI** the model sees few examples of group d and is usually less accurate for it. With high **DPL** the
labels themselves differ between groups, and a model will learn and repeat that pattern.

**Post-training** metrics (DPPL, disparate impact, accuracy difference ...) compare the model's *predictions* between groups, so they need a trained model.

Exam cue: "before training", "in the dataset" → pre-training metrics (CI, DPL). "In the model's predictions" → post-training metrics. Both come from **SageMaker Clarify**; in production, **Model Monitor bias drift** tracks them over time.

### Build the dataset for bias analysis
Join claims with the customer attributes. The **facets** (sensitive attributes) are age group, gender and region.
These columns are not model features, but we still must check the data for them.

In [ ]:
bias_df = claims.merge(policies, on="policy_id", how="left")
bias_df["age_group"] = pd.cut(bias_df["age"], bins=[17, 24, 59, 120],
                              labels=["under_25", "25_to_59", "60_plus"]).astype(str)
bias_df = bias_df[["is_fraud", "age_group", "gender", "region", "claim_amount", "report_delay_days"]]
print({c: bias_df[c].value_counts(normalize=True).round(3).to_dict() for c in ["age_group", "gender", "region"]})

### Two pre-training bias metrics by hand
- **CI** = (n_a − n_d) / (n_a + n_d). `d` = the group we worry about, `a` = everyone else. Range −1..1; near +1 means the group is tiny.
- **DPL** = q_a − q_d, where q = share of the **favourable label** (here `is_fraud = 0`, "not flagged"). Near 0 = both groups are labelled alike.

In [ ]:
def class_imbalance(df, facet, value):
    n_d, n_a = (df[facet] == value).sum(), (df[facet] != value).sum()
    return (n_a - n_d) / (n_a + n_d)

def dpl(df, facet, value, label="is_fraud", favourable=0):
    q_a = (df.loc[df[facet] != value, label] == favourable).mean()
    q_d = (df.loc[df[facet] == value, label] == favourable).mean()
    return q_a - q_d

FACETS = [("age_group", "under_25"), ("gender", "F"), ("region", "West")]
by_hand = pd.DataFrame([{"facet": f, "value": v, "CI": class_imbalance(bias_df, f, v), "DPL": dpl(bias_df, f, v)}
                        for f, v in FACETS]).round(4)
by_hand

Reading the table: under-25 drivers and the West region are **under-represented** (high CI), but the fraud label
rate is almost the same in every group (DPL ≈ 0). So the risk here is too little data for some groups, not unfair labels.

### SageMaker Clarify: upload the dataset
Clarify reads a CSV with a header from S3.

In [ ]:
BIAS_KEY = "safedrive/bias/bias_dataset.csv"
bias_uri = U.write_df(bias_df, BIAS_KEY)
print(bias_uri, bias_df.shape)

### SageMaker Clarify: configure the pre-training bias job
`DataConfig` = where the data is and which column is the label. `BiasConfig` = the favourable label value and the facets.

In [ ]:
import sagemaker
from sagemaker import clarify

sm_session = sagemaker.Session(boto_session=U.session, default_bucket=BUCKET)
BIAS_OUT = "safedrive/bias/pretraining"
data_config = clarify.DataConfig(s3_data_input_path=bias_uri, s3_output_path=U.s3_uri(BIAS_OUT),
                                 label="is_fraud", headers=list(bias_df.columns), dataset_type="text/csv")
bias_config = clarify.BiasConfig(label_values_or_threshold=[0],
                                 facet_name=[f for f, _ in FACETS],
                                 facet_values_or_threshold=[[v] for _, v in FACETS])
print(json.dumps(bias_config.get_config(), indent=1))

### SageMaker Clarify: run the job
A Clarify job is a SageMaker Processing job with the Clarify container. 💰 One `ml.m5.large` for about 6 to 10 minutes.

In [ ]:
clarify_processor = clarify.SageMakerClarifyProcessor(
    role=ROLE, instance_count=1, instance_type="ml.m5.large", sagemaker_session=sm_session,
    job_name_prefix="safedrive-pretraining-bias", tags=U.TAGS)
clarify_processor.run_pre_training_bias(data_config=data_config, data_bias_config=bias_config,
                                        methods=["CI", "DPL", "KL", "JS"], wait=True, logs=False)
print(clarify_processor.latest_job.describe()["ProcessingJobStatus"])

> 🖥️ **Console check:** *Amazon SageMaker AI → Processing → Processing jobs → `safedrive-pretraining-bias-...`*: a Clarify job is an ordinary Processing job with the Clarify container image.
> Then *Amazon S3 → your bucket → `safedrive/bias/pretraining/`*: next to `analysis.json` there is a `report.pdf` (and HTML / notebook versions) with a chart per metric.

### SageMaker Clarify: read the report
The job writes `analysis.json` (plus an HTML / PDF report). Compare CI and DPL with the values computed by hand.
KL and JS measure how different the label distributions of the two groups are (0 = identical).

In [ ]:
analysis = json.loads(U.read_bytes(f"{BIAS_OUT}/analysis.json"))
rows = [{"facet": facet, "value": group["value_or_threshold"], **{m["name"]: m["value"] for m in group["metrics"]}}
        for facet, groups in analysis["pre_training_bias_metrics"]["facets"].items() for group in groups]
pd.DataFrame(rows).round(4)

## 1.3.4 Bias metrics across numeric, text and image data

The same two questions apply to every kind of data:
1. **Representation**: does each group have enough examples? (CI)
2. **Label balance**: does each group get the labels at similar rates? (DPL, KL divergence)

For text and images the "facet" comes from metadata (claim type, language, vehicle colour, camera, lighting ...)
or from a property of the content (text length). After measuring, fix the distribution: collect, reweight or augment.

### Numeric (tabular) data
Already measured above. This is the baseline to compare the other data types with.

In [ ]:
summary = by_hand.assign(data="tabular claims")
summary[["data", "facet", "value", "CI", "DPL"]]

### Text: adjuster notes
Facets: the claim type from the metadata, and a content property (long note or not). Label: `is_fraud`.
Are theft notes rare? Are they labelled fraud more often?

In [ ]:
notes = pd.DataFrame(U.read_jsonl(f"{RAW}/adjuster_notes.jsonl"))
notes["long_note"] = (notes["note"].str.len() > notes["note"].str.len().median()).map({True: "long", False: "short"})
text_facets = [("claim_type", "theft"), ("claim_type", "fire"), ("long_note", "long")]
text_bias = pd.DataFrame([{"data": "adjuster notes", "facet": f, "value": v, "CI": class_imbalance(notes, f, v),
                           "DPL": dpl(notes, f, v)} for f, v in text_facets]).round(4)
print(notes.groupby("claim_type")["is_fraud"].agg(["count", "mean"]).round(3).T)
text_bias

### Images: damage photos
Facet: vehicle colour (a model should not learn "red cars are damaged"). Label: damaged or not.
With only 12 photos the numbers are extreme: that itself is the finding (far too few examples per group).

In [ ]:
manifest = pd.DataFrame(U.read_jsonl(f"{RAW}/images/manifest.jsonl"))
manifest["damaged"] = (manifest["damage_type"] != "none").astype(int)
image_bias = pd.DataFrame([{"data": "damage photos", "facet": "vehicle_color", "value": color,
                            "CI": class_imbalance(manifest, "vehicle_color", color),
                            "DPL": dpl(manifest, "vehicle_color", color, label="damaged", favourable=1)}
                           for color in sorted(manifest["vehicle_color"].unique())]).round(4)
print(pd.crosstab(manifest["vehicle_color"], manifest["damage_type"]))
image_bias

### One view across all three data types
Sort by the size of the representation gap. The biggest |CI| values are where to collect or augment first.

In [ ]:
all_bias = pd.concat([summary, text_bias, image_bias], ignore_index=True)[["data", "facet", "value", "CI", "DPL"]]
all_bias.reindex(all_bias["CI"].abs().sort_values(ascending=False).index)

### KL divergence between a real and a target distribution
KL = 0 means the dataset matches the target. Here the target is "all vehicle colours equally frequent".

In [ ]:
def kl_divergence(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    keep = p > 0
    return float(np.sum(p[keep] * np.log(p[keep] / q[keep])))

observed = manifest["vehicle_color"].value_counts(normalize=True).sort_index()
uniform = np.full(len(observed), 1 / len(observed))
print(observed.round(3).to_dict())
print("KL(observed || uniform) =", round(kl_divergence(observed.values, uniform), 4))

### Optimize the distribution: weights and augmentation targets
Two standard fixes: **reweight** (rare groups count more in the loss) or **augment / collect** until each group
reaches the size of the largest one.

In [ ]:
counts = manifest["vehicle_color"].value_counts().sort_index()
plan = pd.DataFrame({"images": counts,
                     "sample_weight": (len(manifest) / (len(counts) * counts)).round(2),
                     "to_add_for_balance": counts.max() - counts})
weighted = (plan["images"] * plan["sample_weight"]) / (plan["images"] * plan["sample_weight"]).sum()
print("KL after reweighting:", round(kl_divergence(weighted.values, uniform), 4))
plan

## 1.3.5 Resolve class imbalance

| Technique | How | Watch out |
|---|---|---|
| Random undersampling | Drop majority rows | Throws data away |
| Random oversampling | Copy minority rows | Overfits to the copies |
| **SMOTE** | New synthetic minority rows between neighbours | Numeric features only; training set only |
| Class weights | Loss counts minority errors more (`class_weight`, XGBoost `scale_pos_weight`) | No new data needed; often the first choice |
| Augmentation | New text (paraphrase) or images (flip, rotate, brightness) | Must keep the label true |
| Better metric / threshold | Recall, precision, F1, PR-AUC; tune the decision threshold | Accuracy is misleading |

**📘 Concept: Why accuracy misleads on imbalanced data**

Take 1,000 claims with 50 frauds (5%). A "model" that always answers *not fraud* is right 950 times: **95% accuracy**, and it
finds no fraud at all. Accuracy is dominated by the majority class, so use metrics that look at the minority class:

| Metric | Question | Matters most when |
|---|---|---|
| **Recall** = TP / (TP + FN) | Of all real frauds, how many did we catch? | Missing a positive is costly (fraud, disease) |
| **Precision** = TP / (TP + FP) | Of all claims we flagged, how many were fraud? | A false alarm is costly (blocking good customers) |
| **F1** | One number balancing both | You need a single score |
| **PR-AUC** (area under the precision-recall curve) | Quality over all thresholds, focused on the positive class | Positives are rare; ROC-AUC can look good while precision is poor |

How the fixes work: resampling and class weights make minority errors count more during training, so the model predicts
*fraud* more readily. Recall goes up and precision goes down. Moving the **decision threshold** (for example flag at 0.3 instead of 0.5)
has the same kind of effect without retraining.

Two rules: resample **only the training set**, and judge on a validation set with the **real** class ratio.

Exam cue: "99% accuracy but the model misses most positive cases" → imbalance; answer with recall / F1 / PR-AUC plus class weights or resampling.

### Numeric: how imbalanced is the training set?
Load the training file from notebook 1.2a (label first, no header).

In [ ]:
def s3_key(uri):
    return uri.replace(f"s3://{BUCKET}/", "")

cols = [F.LABEL] + state["feature_names"]
train = U.read_csv(s3_key(state["train_s3"]), header=None, names=cols)
val = U.read_csv(s3_key(state["validation_s3"]), header=None, names=cols)
X, y = train[F.FEATURES], train[F.LABEL]
print(y.value_counts().to_dict(), "| fraud share:", round(y.mean(), 4))

### Numeric: resample the training set three ways
Compare the class counts. SMOTE creates **new** rows by interpolating between a fraud row and its nearest fraud neighbours.

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

samplers = {"random undersampling": RandomUnderSampler(random_state=42),
            "random oversampling": RandomOverSampler(random_state=42),
            "SMOTE": SMOTE(random_state=42)}
resampled = {name: s.fit_resample(X, y) for name, s in samplers.items()}
counts = {"original": y.value_counts().to_dict(), **{n: ys.value_counts().to_dict() for n, (_, ys) in resampled.items()}}
pd.DataFrame(counts).T.rename(columns={0: "not fraud", 1: "fraud"})

### Numeric: class weights
No resampling: tell the algorithm that a fraud row counts more. For XGBoost the setting is
`scale_pos_weight = negatives / positives`.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y)
print("balanced class weights:", {c: round(float(w), 2) for c, w in zip([0, 1], weights)})
print("scale_pos_weight for XGBoost:", round((y == 0).sum() / (y == 1).sum(), 1))

### Numeric: does it help?
Train a quick logistic regression four ways and score each on the **untouched** validation set.
Without a fix the model finds few frauds (low recall). Every fix trades precision for recall.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def evaluate(X_train, y_train, **kwargs):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, **kwargs)).fit(X_train, y_train)
    pred = model.predict(val[F.FEATURES])
    return {"train_rows": len(y_train), "precision": precision_score(val[F.LABEL], pred),
            "recall": recall_score(val[F.LABEL], pred)}

runs = {"no fix": evaluate(X, y), "class_weight=balanced": evaluate(X, y, class_weight="balanced")}
runs.update({name: evaluate(Xs, ys) for name, (Xs, ys) in resampled.items()})
pd.DataFrame(runs).T.round(3)

### Text: augment the minority class with a foundation model
Only a few notes belong to fraud claims. The FM writes paraphrases of fraud notes (same facts, new wording).
We do two notes here; in a real project you would loop over all of them and review a sample.

In [ ]:
fraud_notes = notes[notes["is_fraud"] == 1]
print(notes["is_fraud"].value_counts().to_dict())
PROMPT = """Rewrite this motor insurance adjuster note in 2 different ways. Keep every fact, name and number.
Return only a JSON array of 2 strings.

Note: {note}"""
new_rows = []
for row in fraud_notes.head(2).to_dict("records"):
    answer = U.converse(PROMPT.format(note=row["note"]), temperature=0.8)
    try:
        texts = json.loads(answer[answer.find("["): answer.rfind("]") + 1])
    except ValueError:
        texts = []
    new_rows += [{**row, "note": t, "augmented": True} for t in texts]
pd.DataFrame(new_rows)[["claim_id", "is_fraud", "note"]]

### Images: augmentation with Pillow
Small changes that keep the label true: mirror, slight rotation, brightness, zoom.
(A mirrored dent is still a dent. Do not use changes that could alter the label, such as heavy crops.)

In [ ]:
from PIL import Image, ImageEnhance, ImageOps

def augment(img, seed):
    rng = np.random.default_rng(seed)
    out = ImageOps.mirror(img) if rng.random() < 0.5 else img
    out = out.rotate(float(rng.uniform(-8, 8)), fillcolor=(150, 190, 230))
    out = ImageEnhance.Brightness(out).enhance(float(rng.uniform(0.7, 1.3)))
    w, h = out.size
    return ImageOps.fit(out.crop((w * 0.04, h * 0.04, w * 0.96, h * 0.96)), (w, h))

original = Image.open(io.BytesIO(U.read_bytes(f"{RAW}/images/{manifest['file'][0]}"))).convert("RGB")
sheet = Image.new("RGB", (4 * 320, 200))
for i, im in enumerate([original] + [augment(original, s) for s in range(3)]):
    sheet.paste(im.resize((320, 200)), (i * 320, 0))
sheet        # left: original, then three augmented versions

### Images: balance the damage classes
Create augmented copies for every class that has fewer photos than the largest class, and store them in S3
under `safedrive/augmented/images/`.

In [ ]:
target = manifest["damage_type"].value_counts().max()
aug_rows = []
for label, group in manifest.groupby("damage_type"):
    for i in range(target - len(group)):
        src = group.iloc[i % len(group)]
        img = Image.open(io.BytesIO(U.read_bytes(f"{RAW}/images/{src['file']}"))).convert("RGB")
        buf = io.BytesIO()
        augment(img, seed=i).save(buf, format="PNG")
        name = f"aug_{label}_{i:02d}.png"
        s3.put_object(Bucket=BUCKET, Key=f"safedrive/augmented/images/{name}", Body=buf.getvalue())
        aug_rows.append({**src.to_dict(), "file": name, "augmented": True})
balanced = pd.concat([manifest, pd.DataFrame(aug_rows)], ignore_index=True)
pd.DataFrame({"before": manifest["damage_type"].value_counts(), "after": balanced["damage_type"].value_counts()})

## 1.3.2 Label and annotate data

- **SageMaker Ground Truth**: you build the labeling workflow (input manifest → workers → output manifest).
  Workforces: **private** (your staff, Cognito / OIDC), **vendor** (AWS Marketplace), **Amazon Mechanical Turk** (public; not for confidential data).
- Built-in tasks: image classification, bounding boxes, segmentation, text classification, named entities, video, 3D point clouds; custom templates too.
- **Annotation consolidation** combines answers from several workers; **automated data labeling** (active learning) lets a model label the easy items.
- **Ground Truth Plus** = AWS runs the whole labeling project for you. **Amazon A2I** = human review of low-confidence *predictions*.
- GenAI: Ground Truth also supports human evaluation and creation of prompt-response data.

**📘 Concept: Ground Truth workforces and automated data labeling**

**Choosing the workforce** is mostly a data-sensitivity question:

| Workforce | Who | Choose when |
|---|---|---|
| **Private** | Your own employees or contractors (sign in through Amazon Cognito or your identity provider) | Confidential or regulated data, or domain experts are needed (adjusters, doctors) |
| **Vendor** | Third-party labeling companies from AWS Marketplace | You need capacity or special skills under a contract |
| **Amazon Mechanical Turk** | A large public crowd | Non-sensitive data, simple tasks, lowest cost and effort |

**Getting good labels cheaply:**
- Send each item to **several workers**; **annotation consolidation** merges their answers into one label with a confidence score.
- **Automated data labeling** uses *active learning*: humans label a first batch → Ground Truth trains a model on it → the model labels the items it is confident about → only the uncertain items go back to humans → repeat. It pays off on large datasets (thousands of items) and is available for some built-in task types only.

Related services: **Ground Truth Plus** = AWS manages the whole labeling project (turnkey). **Amazon A2I** = human review of a model's low-confidence *predictions* in production, not dataset labeling.

Exam cue: "reduce labeling cost and time for a large dataset" → automated data labeling. "Sensitive data" → private workforce. "No in-house labeling expertise, want AWS to run it" → Ground Truth Plus.

### Input manifest
One JSON line per item with a `source-ref` pointing at the object in S3. This runs without any workforce.

In [ ]:
LABEL_PREFIX = "safedrive/labeling"
input_manifest = [{"source-ref": U.s3_uri(f"{RAW}/images/{f}")} for f in manifest["file"]]
body = "\n".join(json.dumps(r) for r in input_manifest) + "\n"
s3.put_object(Bucket=BUCKET, Key=f"{LABEL_PREFIX}/input.manifest", Body=body.encode())
print(len(input_manifest), "items")
print(body.splitlines()[0])

### Label categories and the worker task template
The category file lists the allowed labels. The template is the HTML page workers see (`crowd-image-classifier`
is a ready-made Ground Truth element).

In [ ]:
LABELS = ["dent", "scratch", "broken_glass", "none"]
categories = {"document-version": "2018-11-28", "labels": [{"label": l} for l in LABELS]}
TEMPLATE = """<script src="https://assets.crowd.aws/crowd-html-elements.js"></script>
<crowd-form>
  <crowd-image-classifier name="crowd-image-classifier" header="What damage does the car show?"
      src="{{ task.input.taskObject | grant_read_access }}" categories="{{ task.input.labels | to_json | escape }}">
    <full-instructions header="Instructions">Choose the one damage type you can see.</full-instructions>
    <short-instructions>Choose the damage type.</short-instructions>
  </crowd-image-classifier>
</crowd-form>"""
s3.put_object(Bucket=BUCKET, Key=f"{LABEL_PREFIX}/categories.json", Body=json.dumps(categories).encode())
s3.put_object(Bucket=BUCKET, Key=f"{LABEL_PREFIX}/template.liquid", Body=TEMPLATE.encode())
print(U.list_keys(LABEL_PREFIX + "/"))

### 🔒 Create the labeling job
Prerequisite: a **private workforce and work team** (SageMaker console → Ground Truth → Labeling workforces → Private;
invite yourself by e-mail), the team ARN in the environment variable `SAFEDRIVE_WORKTEAM_ARN`, and a CORS rule on the bucket
(the console adds it when you create a job there). Someone must then log in to the worker portal and label the 12 images.
The two Lambda ARNs are AWS-owned functions for the built-in image classification task (account ID differs per Region).

In [ ]:
sm = U.client("sagemaker")
WORKTEAM_ARN = os.environ.get("SAFEDRIVE_WORKTEAM_ARN", "")
GT_ACCOUNTS = {"us-east-1": "432418664414", "us-east-2": "266458841044", "us-west-2": "081040173940",
               "eu-west-1": "568282634449", "eu-west-2": "487402164563", "eu-central-1": "203001061592",
               "ap-south-1": "565803892007", "ap-southeast-1": "377565633583", "ap-southeast-2": "454466003867",
               "ap-northeast-1": "477331159723", "ap-northeast-2": "845288260483", "ca-central-1": "918755190332"}
gt_lambda = f"arn:aws:lambda:{U.REGION}:{GT_ACCOUNTS.get(U.REGION, 'ACCOUNT')}:function"
LABEL_JOB = f"safedrive-damage-labels-{int(time.time())}"
print("work team:", WORKTEAM_ARN or "(not set)", "|", gt_lambda)

In [ ]:
if U.expensive("Ground Truth labeling job - needs a private work team") and WORKTEAM_ARN:
    sm.create_labeling_job(
        LabelingJobName=LABEL_JOB, LabelAttributeName="damage-type", RoleArn=ROLE, Tags=U.TAGS,
        InputConfig={"DataSource": {"S3DataSource": {"ManifestS3Uri": U.s3_uri(f"{LABEL_PREFIX}/input.manifest")}}},
        OutputConfig={"S3OutputPath": U.s3_uri(f"{LABEL_PREFIX}/output/")},
        LabelCategoryConfigS3Uri=U.s3_uri(f"{LABEL_PREFIX}/categories.json"),
        HumanTaskConfig={
            "WorkteamArn": WORKTEAM_ARN,
            "UiConfig": {"UiTemplateS3Uri": U.s3_uri(f"{LABEL_PREFIX}/template.liquid")},
            "PreHumanTaskLambdaArn": f"{gt_lambda}:PRE-ImageMultiClass",
            "AnnotationConsolidationConfig": {"AnnotationConsolidationLambdaArn": f"{gt_lambda}:ACS-ImageMultiClass"},
            "TaskTitle": "Classify car damage", "TaskDescription": "Choose the damage type shown in the photo",
            "NumberOfHumanWorkersPerDataObject": 1, "TaskTimeLimitInSeconds": 300,
            "TaskAvailabilityLifetimeInSeconds": 86400})
    print("created", LABEL_JOB)

🔒 Check the job. `LabelCounters` shows how many items are labelled so far; the job completes when workers finish.

In [ ]:
if U.expensive("Ground Truth labeling job status") and WORKTEAM_ARN:
    job = sm.describe_labeling_job(LabelingJobName=LABEL_JOB)
    print(job["LabelingJobStatus"], job["LabelCounters"])
    print(job.get("LabelingJobOutput", {}).get("OutputDatasetS3Uri", "no output manifest yet"))

> 🖥️ **Console check:** If you ran this 🔒 cell: *Amazon SageMaker AI → Ground Truth → Labeling jobs → `safedrive-damage-labels-...`*.
> Look at the progress counter and, after completion, the labelled images. *Labeling workforces → Private* has the link to the worker portal where the tasks appear.

### What the output manifest looks like
Ground Truth returns the input line plus the label and its metadata (an **augmented manifest**).
SageMaker training jobs can read this format directly. We build the same structure from our known labels.

In [ ]:
def output_line(row):
    return {"source-ref": U.s3_uri(f"{RAW}/images/{row['file']}"),
            "damage-type": LABELS.index(row["damage_type"]),
            "damage-type-metadata": {"class-name": row["damage_type"], "confidence": 1.0, "type": "groundtruth/image-classification",
                                     "human-annotated": "yes", "job-name": "labeling-job/safedrive-damage-labels"}}

output_manifest = [output_line(r) for r in manifest.to_dict("records")]
body = "\n".join(json.dumps(r) for r in output_manifest) + "\n"
s3.put_object(Bucket=BUCKET, Key=f"{LABEL_PREFIX}/output-example.manifest", Body=body.encode())
print(json.dumps(output_manifest[0], indent=1))

## 1.3.6 Validate AI training-data integrity

A fine-tuned model repeats what is in its training pairs. Check **before** training:

| Check | Catches | Tool here |
|---|---|---|
| Schema and types | Wrong keys, broken JSON | Python |
| Empty prompt or answer | Useless pairs | Python |
| Length | Pairs over the token limit | Python |
| Duplicates | Over-weighted pairs, train/validation leakage | Python |
| PII | Personal data memorised by the model | Amazon Comprehend |
| Harmful / toxic content | Unsafe model behaviour | Bedrock Guardrails `ApplyGuardrail` |
| Does the answer fit the question? | Mismatched or wrong pairs | LLM-as-a-judge, human review |

### Load the raw pairs
Notebook 1.2b converted this file to the Bedrock format with structural checks only. Here every record is inspected.

In [ ]:
qa_raw = U.read_jsonl(f"{RAW}/finetune_qa.jsonl")
report = pd.DataFrame({"record": range(len(qa_raw)), "issue": ""})
print(len(qa_raw), "records")
qa_raw[-1]

### Rule checks: schema, empty fields, length
Each record gets the first rule it breaks.

In [ ]:
MAX_CHARS = 4000      # about 1,000 tokens per field

def rule_check(r):
    if set(r) != {"prompt", "completion"}:
        return "wrong schema"
    if not all(isinstance(r[k], str) and r[k].strip() for k in r):
        return "empty field"
    if max(len(r["prompt"]), len(r["completion"])) > MAX_CHARS:
        return "too long"
    return ""

report["issue"] = [rule_check(r) for r in qa_raw]
report[report["issue"] != ""]

### Duplicate pairs
Compare a normalized form (lower case, single spaces) so that trivial differences do not hide a duplicate.

In [ ]:
def norm(r):
    return re.sub(r"\s+", " ", (r.get("prompt", "") + " || " + r.get("completion", "")).lower()).strip()

seen = set()
for i, r in enumerate(qa_raw):
    if report.at[i, "issue"] == "":
        if norm(r) in seen:
            report.at[i, "issue"] = "duplicate"
        seen.add(norm(r))
report[report["issue"] == "duplicate"]

### PII check with Amazon Comprehend
Two practical points: only **sensitive** entity types matter (Comprehend also tags dates), and public company
contact details are on an **allow-list** (the SafeDrive hotline is not personal data).

In [ ]:
comp = U.client("comprehend")
SENSITIVE = {"NAME", "PHONE", "EMAIL", "SSN", "ADDRESS", "CREDIT_DEBIT_NUMBER", "BANK_ACCOUNT_NUMBER"}
ALLOWED = {"1-800-555-0100", "safedrive"}

def find_pii(text):
    found = comp.detect_pii_entities(Text=text, LanguageCode="en")["Entities"]
    hits = [(e["Type"], text[e["BeginOffset"]:e["EndOffset"]]) for e in found
            if e["Type"] in SENSITIVE and e["Score"] > 0.8]
    return [h for h in hits if h[1].lower() not in ALLOWED]

for i, r in enumerate(qa_raw):
    if report.at[i, "issue"] == "":
        pii = find_pii(r["prompt"] + "\n" + r["completion"])
        if pii:
            report.at[i, "issue"] = "PII: " + ", ".join(sorted({t for t, _ in pii}))
report[report["issue"].str.startswith("PII")]

### Content safety: the shared SafeDrive guardrail
`C.ensure_guardrail()` returns the guardrail of the claims assistant (PII rules and one denied topic).
`ApplyGuardrail` checks any text **without calling a model**. Try it on the PII record and on a harmless one.

In [ ]:
brt = U.client("bedrock-runtime")
guardrail_id, guardrail_version = C.ensure_guardrail()

def screen(text, gid, version):
    """Return (action, list of things the guardrail found)."""
    resp = brt.apply_guardrail(guardrailIdentifier=gid, guardrailVersion=version, source="OUTPUT",
                               content=[{"text": {"text": text}}])
    hits = []
    for a in resp["assessments"]:
        hits += [f["type"] for f in a.get("contentPolicy", {}).get("filters", [])]
        hits += [p["type"] for p in a.get("sensitiveInformationPolicy", {}).get("piiEntities", [])]
        hits += [t["name"] for t in a.get("topicPolicy", {}).get("topics", [])]
    return resp["action"], hits

pii_index = int(report.index[report["issue"].str.startswith("PII")][0])
print(screen(qa_raw[pii_index]["completion"], guardrail_id, guardrail_version))
print(screen(qa_raw[1]["completion"], guardrail_id, guardrail_version))

### Content safety: a screening guardrail with content filters
The assistant's guardrail has no toxicity filters, so we create a small one just for screening datasets:
**content filters** for insults, hate, sexual content and violence. Reused if it already exists.

In [ ]:
bedrock = U.client("bedrock")
SCREEN_NAME = "safedrive-data-screen"
found = [g for g in bedrock.list_guardrails(maxResults=100)["guardrails"] if g["name"] == SCREEN_NAME]
if found:
    screen_id = found[0]["id"]
else:
    screen_id = bedrock.create_guardrail(
        name=SCREEN_NAME, description="Screens training data for harmful content",
        blockedInputMessaging="blocked", blockedOutputsMessaging="blocked",
        contentPolicyConfig={"filtersConfig": [
            {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "SEXUAL", "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
            {"type": "VIOLENCE", "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"}]},
        tags=[{"key": t["Key"], "value": t["Value"]} for t in U.TAGS])["guardrailId"]
U.wait_until(lambda: bedrock.get_guardrail(guardrailIdentifier=screen_id)["status"],
             ok=("READY",), bad=("FAILED",), delay=3, label="guardrail")

> 🖥️ **Console check:** *Amazon Bedrock → Guardrails → `safedrive-data-screen`* (working draft).
> Look at the content filters and their strengths. The test panel on the right lets you paste a sentence and see which filter fires.

### Content safety: screen every remaining pair
The working draft of a guardrail is addressed as version `DRAFT`. Records where the guardrail intervenes are marked.

In [ ]:
for i, r in enumerate(qa_raw):
    if report.at[i, "issue"] == "":
        action, hits = screen(r["prompt"] + "\n" + r["completion"], screen_id, "DRAFT")
        if action == "GUARDRAIL_INTERVENED":
            report.at[i, "issue"] = "unsafe: " + ", ".join(sorted(set(hits)))
report[report["issue"].str.startswith("unsafe")]

### Prompt-response fit: LLM as a judge
Rules cannot tell whether an answer really answers its question. A small, cheap model can give a first opinion.
This check only **flags** pairs for human review; it does not remove them.

In [ ]:
JUDGE = """Does the answer address the question? Reply with only YES or NO.
Question: {prompt}
Answer: {completion}"""
verdicts = {}
for i, r in enumerate(qa_raw):
    if report.at[i, "issue"] == "":
        verdicts[i] = U.converse(JUDGE.format(**r), model_id=U.SMALL_MODEL, max_tokens=5, temperature=0).strip().upper()
print(pd.Series(verdicts).str[:3].value_counts().to_dict())
print("to review:", [qa_raw[i]["prompt"] for i, v in verdicts.items() if not v.startswith("YES")])

### The integrity report
Every record with its verdict. Seven bad records were planted in the file; all should be caught.

In [ ]:
report["prompt"] = [str(r.get("prompt", r))[:50] for r in qa_raw]
print(report["issue"].replace("", "ok").str.split(":").str[0].value_counts().to_dict())
report[report["issue"] != ""]

### Write the cleaned file back
Keep the clean pairs, convert them to the Bedrock conversation format (same as notebook 1.2b) and overwrite
`train.jsonl`. `finetune_train_s3` now points at data that passed every check.

In [ ]:
SYSTEM = "You are the SafeDrive Insurance claims assistant. Answer briefly and only from SafeDrive policy."
good = [r for i, r in enumerate(qa_raw) if report.at[i, "issue"] == ""]
rows = [{"schemaVersion": "bedrock-conversation-2024", "system": [{"text": SYSTEM}],
         "messages": [{"role": "user", "content": [{"text": r["prompt"]}]},
                      {"role": "assistant", "content": [{"text": r["completion"]}]}]} for r in good]
FT_KEY = "safedrive/genai/finetune/train.jsonl"
s3.put_object(Bucket=BUCKET, Key=FT_KEY, Body=("\n".join(json.dumps(r) for r in rows) + "\n").encode())
print(len(qa_raw), "raw records ->", len(rows), "clean records ->", U.s3_uri(FT_KEY))

## Exam traps

- **Pre-training bias metrics (CI, DPL, KL, JS) need no model**; post-training metrics (DPPL, DI, accuracy difference) do. Read which one the question asks for.
- **SMOTE / oversampling before the split** leaks synthetic copies into validation and test. Split first, resample only the training part.
- With 1% positives, **99% accuracy means nothing**. Look for recall, precision, F1 or PR-AUC in the answers.
- **Mean imputation with outliers** is a trap; use the median. Dropping rows is fine only when few are affected.
- **Mechanical Turk is a public workforce**: not for confidential or regulated data; use a private or vendor workforce.

## Quiz

**1.** A nightly Glue ETL job loads claims into a data lake. The team wants the job to fail when more than 1%
of `claim_amount` values are missing, with the least custom code. What should they use?
- A. A Lambda function that scans the output with pandas
- B. AWS Glue Data Quality rules (DQDL) evaluated in the job
- C. Amazon Macie
- D. SageMaker Model Monitor

<details><summary>Answer</summary>B. A DQDL rule such as <code>Completeness "claim_amount" > 0.99</code> runs inside Glue and can stop the job.</details>

**2.** A loan dataset has 90% of rows from applicants older than 40. Before training, the ML engineer must
quantify this. Which SageMaker Clarify metric measures it?
- A. Difference in positive proportions in predicted labels (DPPL)
- B. Class imbalance (CI)
- C. SHAP values
- D. Disparate impact (DI)

<details><summary>Answer</summary>B. CI is a pre-training metric for how unevenly a facet's groups are represented. DPPL and DI need model predictions.</details>

**3.** A fraud dataset has 0.5% positive examples. An engineer applies SMOTE to the full dataset, then splits
it into train and test, and reports excellent test recall. What is wrong?
- A. SMOTE cannot be used with numeric features
- B. Synthetic samples derived from test rows leaked into training; resample only the training split
- C. The split should not be stratified
- D. Recall is the wrong metric for imbalanced data

<details><summary>Answer</summary>B. Resampling before the split causes leakage and an unrealistic test distribution.</details>

**4.** A hospital needs 50,000 medical images labelled. The images contain patient data and only employed
radiologists may see them. Which Ground Truth workforce is appropriate?
- A. Amazon Mechanical Turk
- B. A private workforce
- C. Automated labeling only, with no humans
- D. Any vendor from AWS Marketplace without a contract

<details><summary>Answer</summary>B. A private workforce (the company's own staff) keeps confidential data in-house.</details>

**5.** Before fine-tuning a Bedrock model on 20,000 support conversations, a team must make sure the
dataset has no abusive language and no customer phone numbers. Which combination fits best?
- A. Amazon Comprehend PII detection and Bedrock Guardrails content filters applied to the dataset
- B. Amazon Rekognition moderation labels
- C. AWS WAF rules on the S3 bucket
- D. Increasing the number of training epochs

<details><summary>Answer</summary>A. Comprehend finds PII in text; the <code>ApplyGuardrail</code> API screens text for harmful content without invoking a model.</details>

## Save state
`claims_v2_clean_s3` was saved in section 1.3.7. Here the fine-tuning file is marked as cleaned.

In [ ]:
saved = U.save_state(finetune_train_s3=U.s3_uri(FT_KEY), finetune_cleaned=True)
{k: saved.get(k) for k in ("claims_v2_clean_s3", "finetune_train_s3", "finetune_cleaned")}

## Cleanup

**Kept for later notebooks** (no idle cost): Glue database `safedrive` and table `claims_v2`, the cleaned files in S3,
the shared guardrail `safedrive-guardrail`. Nothing created here bills while idle; the cells below just tidy up.

### Before you delete: what later notebooks use

The cleanup cells below **do not delete** the items in this table.

| Kept | Used by later notebooks | If you delete it anyway |
|---|---|---|
| Shared guardrail `safedrive-guardrail` | 3.2b, 4.3b | Rebuilt by the catch-up cell |
| Cleaned fine-tuning file | 2.2b | 2.2b falls back to a simple clean of the raw file |
| Glue table `claims_v2` and the cleaned claims file | Not used again | - |

- **Continuing to the next notebook soon?** Run the cleanup cells and keep the items above. None of them bills while idle (only a little storage).
- **Stopping for good?** Run `99_cleanup/99_teardown.ipynb`; it removes everything, including these.

Delete the Glue Data Quality ruleset.

In [ ]:
U.safe(glue.delete_data_quality_ruleset, Name=RULESET)

Delete the DataBrew profile job.

In [ ]:
U.safe(brew.delete_job, Name=PROFILE_JOB)

Delete the DataBrew dataset.

In [ ]:
U.safe(brew.delete_dataset, Name=BREW_DS)

Delete the screening guardrail (the shared `safedrive-guardrail` stays).

In [ ]:
U.safe(bedrock.delete_guardrail, guardrailIdentifier=screen_id)

🔒 Stop the labeling job if you started one.

In [ ]:
if U.expensive("stop labeling job") and WORKTEAM_ARN:
    U.safe(sm.stop_labeling_job, LabelingJobName=LABEL_JOB)